# Async Retrievers
> Run retrieval concurrently with async/await for I/O-heavy workloads.

| Module | `anchor.retrieval` |
|--------|--------------------|
| Key classes | `AsyncDenseRetriever`, `AsyncHybridRetriever` |
| Difficulty | Intermediate |

`AsyncDenseRetriever` and `AsyncHybridRetriever` expose `await aretrieve()`
for non-blocking retrieval in async applications.

**Time:** ~7 minutes

## Setup

In [ ]:
import asyncio
from anchor.retrieval import AsyncDenseRetriever, AsyncHybridRetriever
from anchor.storage import InMemoryVectorStore, InMemoryContextStore
from anchor.models import ContextItem, SourceType, QueryBundle

## Shared Infrastructure

In [ ]:
vector_store = InMemoryVectorStore()
ctx_store = InMemoryContextStore()

def embed_fn(text: str) -> list[float]:
    padded = text[:128].ljust(128)
    return [hash(c) % 100 / 100.0 for c in padded]

items = [
    ContextItem(id=f"async-{i}", content=f"Async doc {i}: {topic} programming guide.",
               source=SourceType.RETRIEVAL, score=0.0, priority=5, token_count=10)
    for i, topic in enumerate(["python", "rust", "go", "java", "kotlin"])
]

print(f"Prepared {len(items)} items for async retrieval.")

## AsyncDenseRetriever
Index and retrieve using `await aretrieve()`.

In [ ]:
async_dense = AsyncDenseRetriever(
    vector_store=vector_store,
    context_store=ctx_store,
    embed_fn=embed_fn,
)

# Index is typically synchronous
async_dense.index(items)

print(f"Indexed {len(items)} items in AsyncDenseRetriever.")

## Run an Async Query

In [ ]:
async def run_dense_query():
    query = QueryBundle(query_str="programming guide")
    results = await async_dense.aretrieve(query, top_k=3)
    print(f"Async dense results ({len(results)}):")
    for i, item in enumerate(results):
        print(f"  [{i+1}] {item.content[:50]}")
    return results

# In a notebook, use 'await' directly. Here we use asyncio.run().
results = asyncio.run(run_dense_query())

## AsyncHybridRetriever
Combine async dense and sparse retrieval with concurrent execution.

In [ ]:
from anchor.retrieval import SparseRetriever

sparse = SparseRetriever(context_store=ctx_store)
sparse.index(items)

async_hybrid = AsyncHybridRetriever(
    dense=async_dense,
    sparse=sparse,
    weights=(0.6, 0.4),
)

print("AsyncHybridRetriever created with weights (0.6, 0.4).")

## Concurrent Retrieval
Run multiple queries in parallel using `asyncio.gather`.

In [ ]:
async def run_concurrent_queries():
    queries = [
        QueryBundle(query_str="python programming"),
        QueryBundle(query_str="rust guide"),
        QueryBundle(query_str="concurrent systems"),
    ]

    tasks = [async_hybrid.aretrieve(q, top_k=2) for q in queries]
    all_results = await asyncio.gather(*tasks)

    for q, results in zip(queries, all_results):
        top = results[0].content[:40] if results else "(none)"
        print(f"  '{q.query_str}' -> {top}")

asyncio.run(run_concurrent_queries())

## Key Takeaways
- `AsyncDenseRetriever` and `AsyncHybridRetriever` use `await aretrieve()`
- Indexing remains synchronous; retrieval is async for I/O concurrency
- Use `asyncio.gather` to run multiple queries in parallel
- Ideal for web servers and applications with concurrent request handling

**Next:** [Late Interaction Retriever](06_late_interaction.ipynb)